# RNN に基づく言語モデル

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import collections
import random
import json

! pip install mecab-python3 unidic-lite
import MeCab

In [ ]:
# load data 
def remove_punctuation(text):
    # 句読点を削除する関数
    punctuation = "，．、。！？「」（）『』【】《》〈〉…"
    for p in punctuation:
        text = text.replace(p, "")
    return text


def read_dataset(file_path):
    # 複数の行に分かれた日本語文を読み込み，単語ごとに分割してリストに格納し，generatorで返す
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # MeCabを用いて単語に分割
            mecab = MeCab.Tagger("-Owakati")
            words = mecab.parse(line)
            words = remove_punctuation(words).strip().split()
            yield words

# データセットの読み込み
data = list(read_dataset("data-rnnlm.txt"))
print(data[:2])

In [ ]:
# 単語をIDに変換するための辞書を作成
word2id = collections.defaultdict(lambda: len(word2id))
SENT_BOUNDARY = word2id["<S>"] # 文の開始を表す特殊なトークン
UNK = word2id["<UNK>"] # 語彙に存在しない単語を表す特殊なトークン

for sentence in data:
    for word in sentence:
        word2id[word] # 単語をIDに変換してword2idに登録

# 存在しない単語はUNKにマッピング
word2id = collections.defaultdict(lambda: UNK, word2id) 
# 保存用に IDから単語へのマッピングも作成
id2word = {i: w for w, i in word2id.items()}

# 語彙のサイズを取得
vocab_size = len(word2id)

# 辞書の保存
with open("word2id.json", "w", encoding="utf-8") as f:
    json.dump(dict(word2id), f, ensure_ascii=False, indent=4)


In [ ]:
# モデルの定義
class SimpleRNNLM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        """シンプルなRNNベースの言語モデル
        Args:
            vocab_size: 語彙のサイズ
            embed_size: 単語埋め込みの次元数
            hidden_size: RNNの隠れ状態の次元数
        """

        super(SimpleRNNLM, self).__init__()
        # 単語IDを固定次元のベクトルに変換
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # RNNレイヤー
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        # RNNの出力を語彙サイズに変換して、次の単語を予測
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        # x: (batch_size, seq_len)
        x = self.embedding(x) # (batch_size, seq_len, embed_size)
        out, h = self.rnn(x, h) # out: (batch_size, seq_len, hidden_size)
        out = self.fc(out) # (batch_size, seq_len, vocab_size)
        return out, h

In [ ]:
# ハイパーパラメータの設定
EMBED_SIZE = 64
HIDDEN_SIZE = 128
LEARNING_RATE = 0.01
EPOCHS = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用中のデバイス: {device}")

In [ ]:
model = SimpleRNNLM(vocab_size, EMBED_SIZE, HIDDEN_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# データをテンソルに変換
data_tensors = []
for sentence in data: 
    # 文の開始終了トークンを追加
    ids = [SENT_BOUNDARY] + [word2id[word] for word in sentence] + [SENT_BOUNDARY] # 文の終了トークンも追加
    data_tensors.append(torch.tensor(ids, dtype=torch.long))

In [ ]:
# 学習ループ
for epoch in range(EPOCHS):
    total_loss = 0
    for ids in data_tensors:
        # 入力は文の最初から最後から2番目まで
        # ターゲット（正解）は2番目から最後（次の単語）まで
        input_data = ids[:-1].unsqueeze(0).to(device) # (1, seq_len-1)
        target_data = ids[1:].unsqueeze(0).to(device)  # (1, seq_len-1)

        # 隠れ状態の初期化
        h = None 

        # 勾配の初期化
        optimizer.zero_grad()

        # 順伝播
        output, h = model(input_data, h)

        # 損失の計算 (出力を[要素数, 語彙サイズ]、ターゲットを[要素数]に変形)
        loss = criterion(output.view(-1, vocab_size), target_data.view(-1))
        
        # 逆伝播と最適化
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(data_tensors):.4f}")

print("done.")

In [ ]:
# 推論してみる
words = ["<S>", "私", "は"] # 文の開始トークンからスタート

# SENT_BOUNDARYを予測するまで単語を生成
while True:
    # wordsをIDに変換してテンソルにする
    input_ids = torch.tensor([[word2id[word] for word in words]], dtype=torch.long) # (1, seq_len)
    
    # 隠れ状態の初期化
    h = None
    with torch.no_grad():
        output, h = model(input_ids, h) # output: (1, seq_len, vocab_size)
    
    # 最後の単語の予測を取得
    last_output = output[0, -1] # (vocab_size,)
    
    # 各単語の予測確率を計算し出力（トップ3）
    probs = torch.softmax(last_output, dim=0)
    top3_probs, top3_indices = torch.topk(probs, 3)
    for i, prob in zip(top3_indices, top3_probs):
        print(f"{id2word[i.item()]:s}: {prob.item():.4f}", end=", ")
    
    # 予測された単語IDを取得し、単語に変換
    predicted_id = torch.argmax(last_output).item() # 予測された単語ID
    predicted_word = id2word[predicted_id] # 予測された単語
    print("-->", predicted_word)

    # 予測された単語を文に追加
    words.append(predicted_word)

    # 文の終了トークンが予測されたら終了
    if predicted_word == "<S>":
        break

# 文の開始と終了トークンを除いて表示
print("生成された文:", "".join(words[1:-1])) 